## Working with Batches

The Batch API endpoint allows users to submit requests for asynchronous batch processing. We will process these requests within 24 hours. The details of each request will be read from a pre-uploaded file, and the responses will be written to an output file. You can query the batch object for status updates and results. Each model will be offered at 50% cost discount vs. the synchronous APIs. 



# Univeral Code Used for the Entire Notebook

Let's set up our libraries and client

In [1]:
# Import the OpenAI class from the openai module
from openai import OpenAI
from dotenv import load_dotenv
# Import the time module to allow for time-related functions
import time
load_dotenv()

True

In [2]:
# Create a client we can use
client = OpenAI()

## Preparing the Batch File

Batches start with a .jsonl file where each line contains the details of an individual request to the API. For now, the available endpoints are /v1/chat/completions (Chat Completions API) and /v1/embeddings (Embeddings API). For a given input file, the parameters in each line's body field are the same as the parameters for the underlying endpoint. Each request must include a unique custom_id value, which you can use to reference results after completion. Here's an example of an input file with 2 requests. Note that each input file can only include requests to a single model.

NOTE: For some insane reason you are required to indicate the API endpoint here and in the batch creation later on. Presumably, OpenAI will one day allow hitting multiple different APIs in one batch request; but today is not that day. 

<br/>
Examples:

```
{"custom_id": "request-1", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4o", "messages": [{"role": "system", "content": "You are a helpful assistant."},{"role": "user", "content": "Give me three paragraphs on the penguin lifecycle."}],"max_tokens": 1000}}

{"custom_id": "request-2", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4o", "messages": [{"role": "system", "content": "You are a helpful assistant."},{"role": "user", "content": "Give me three paragraphs on penguin mating habits."}],"max_tokens": 1000}}

{"custom_id": "request-3", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "gpt-4o", "messages": [{"role": "system", "content": "You are a helpful assistant."},{"role": "user", "content": "Give me three paragraphs on penguin species differences."}],"max_tokens": 1000}}
```

In [3]:
prompt_template = """
You will review an error case from an LLM that predicted the T stage in a breast‑cancer pathology report. Read:
- the pathology text
- the true T stage (based on AJCC 7th edition)
- the model’s prediction and reasoning,

Choose the single best cause of the model’s error from the list below.
<Error categories>
1.Incorrect Information Extraction – the model missed or mis‑read key facts in the report.
2.Incorrect Inference – it saw the right facts but drew the wrong conclusion.
    * 2a Numerical Incompetence – wrong comparison or arithmetic (e.g., reading 1.9 cm as “> 2 cm”).
    * 2b Incorrect Knowledge – uses the wrong AJCC rule (e.g., says 1.8 cm → T2, which is false). To label an error as 2b, the model must explicitly state a clearly incorrect AJCC rule in its reasoning. If not, label it as 2 (Incorrect Inference).
3.Conflicting Ground Truth – the provided label contradicts the pathology text.
4.Incomplete Information – the report itself lacks the detail needed to assign the given T stage.
(so, 3 and 4 are the issue of the dataset itself)

—————

Pathological Report:
{text}


Ground Truth T Stage ((based on AJCC 7th edition)):
{ground_truth}


Model’s Prediction:
{pred}


Model’s Reasoning:
{reasoning}
"""

In [4]:
import json
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd
import openai
from openai.lib._pydantic import to_strict_json_schema

class ErrorAnalysisResponse(BaseModel):
    error_cause: Literal[
        "Incorrect Information Extraction",
        "Incorrect Inference",
        "Numerical Incompetence",
        "Incorrect Knowledge",
        "Conflicting Ground Truth",
        "Incomplete Information"
    ] = Field(
        ...,
        description="The single best cause of the model's error."
    )
    reason: str = Field(
        ...,
        description="A concise explanation referencing the model’s reasoning and report, justifying the selected error cause."
    )


schema = {"type": "json_schema",
           "json_schema": {"name": "ErrorAnalysisResponse",
                           "schema": to_strict_json_schema(ErrorAnalysisResponse),
                           }
          }

schema

{'type': 'json_schema',
 'json_schema': {'name': 'ErrorAnalysisResponse',
  'schema': {'properties': {'error_cause': {'description': "The single best cause of the model's error.",
     'enum': ['Incorrect Information Extraction',
      'Incorrect Inference',
      'Numerical Incompetence',
      'Incorrect Knowledge',
      'Conflicting Ground Truth',
      'Incomplete Information'],
     'title': 'Error Cause',
     'type': 'string'},
    'reason': {'description': 'A concise explanation referencing the model’s reasoning and report, justifying the selected error cause.',
     'title': 'Reason',
     'type': 'string'}},
   'required': ['error_cause', 'reason'],
   'title': 'ErrorAnalysisResponse',
   'type': 'object',
   'additionalProperties': False}}}

In [24]:
import json
from pydantic import BaseModel, Field
from typing import Literal
import pandas as pd
from copy import deepcopy


def create_openai_batch_file(requests_data, output_file_path="batch_input.jsonl"):
    with open(output_file_path, 'w') as f:
        for request in requests_data:
            json_string = json.dumps(request)
            f.write(json_string + '\n')
    print(f"Batch file created successfully at: {output_file_path}")
    return output_file_path


requests = []

example_entry = {"method": "POST", "url": "/v1/chat/completions", "body": 
                                                {"model": "o3-2025-04-16",
                                                 "messages": [{"role": "system", "content": "You are a helpful assistant."},],
                                                 "response_format": schema}
                }

# /home/yl3427/cylab/selfCorrectionAgent/t_zscot_only_errors_vs_kewltm.csv
# patient_filename,t,text,canon_truth,zscot_t_stage,zscot_t_reasoning,is_error

# /home/yl3427/cylab/selfCorrectionAgent/t_kewltm_only_errors_vs_zscot.csv
# patient_filename,t,text,canon_truth,cmem_t_40reports_ans_str,cmem_t_40reasoning,is_error

# /home/yl3427/cylab/selfCorrectionAgent/t_rag_only_errors_vs_kewrag.csv
# patient_filename,t,text,canon_truth,rag_raw_t_stage,rag_raw_t_reasoning,is_error


# /home/yl3427/cylab/selfCorrectionAgent/t_kewrag_only_errors_vs_rag.csv
# patient_filename,t,text,canon_truth,ltm_rag1_t_stage,ltm_rag1_t_reasoning,is_error

df = pd.read_csv("/home/yl3427/cylab/selfCorrectionAgent/t_kewrag_only_errors_vs_rag.csv")
for index, row in df.iterrows():
    entry = deepcopy(example_entry)
    patient_filename = row['patient_filename']
    text = row['text']
    ground_truth = f"T{row['t']+1}"

    ## change
    pred = row["ltm_rag1_t_stage"]
    reasoning = row["ltm_rag1_t_reasoning"]
    ##


    user_prompt = prompt_template.format(
        text=text,
        ground_truth=ground_truth,
        pred=pred,
        reasoning=reasoning
    )
    entry["custom_id"] = f"request-{patient_filename}-{index}"
    entry["body"]["messages"].append({"role": "user", "content": user_prompt})
    requests.append(entry)


create_openai_batch_file(requests, output_file_path="t_kewrag_only_errors_vs_rag.jsonl")


Batch file created successfully at: t_kewrag_only_errors_vs_rag.jsonl


't_kewrag_only_errors_vs_rag.jsonl'

### Uploading the File

After creating your batch file, you must upload it so that you can reference it correctly when kicking off batches. Upload your .jsonl file using the Files API.

In [26]:
import json

with open("t_kewrag_only_errors_vs_rag.jsonl", "r") as f:
    lines = f.readlines()
first_entry = json.loads(lines[1])
first_entry

{'method': 'POST',
 'url': '/v1/chat/completions',
 'body': {'model': 'o3-2025-04-16',
  'messages': [{'role': 'system', 'content': 'You are a helpful assistant.'},
   {'role': 'user',
    'content': "\nYou will review an error case from an LLM that predicted the T stage in a breast‑cancer pathology report. Read:\n- the pathology text\n- the true T stage (based on AJCC 7th\u202fedition)\n- the model’s prediction and reasoning,\n\nChoose the single best cause of the model’s error from the list below.\n<Error categories>\n1.Incorrect Information Extraction – the model missed or mis‑read key facts in the report.\n2.Incorrect Inference – it saw the right facts but drew the wrong conclusion.\n    * 2a\xa0Numerical Incompetence – wrong comparison or arithmetic (e.g., reading 1.9\u202fcm as “>\u202f2\u202fcm”).\n    * 2b\xa0Incorrect Knowledge – uses the wrong AJCC rule (e.g., says 1.8\u202fcm → T2, which is false). To label an error as 2b, the model must explicitly state a clearly incorrect 

In [30]:
resulting_files = [
    # "t_zscot_only_errors_vs_kewltm.jsonl",
                #   "t_kewltm_only_errors_vs_zscot.jsonl",
                #   "t_rag_only_errors_vs_kewrag.jsonl",
                  "t_kewrag_only_errors_vs_rag.jsonl"
                  ]
for file_name in resulting_files:
    batch_input_file = client.files.create(
        file=open(file_name, "rb"), 
        purpose="batch"  
    )

    batch_input_file_id = batch_input_file.id

    batch = client.batches.create(
        input_file_id=batch_input_file_id, 
        endpoint="/v1/chat/completions", 
        completion_window="24h", 
        metadata={
            "description": file_name+"_3"  
        }
    )
    print(batch)
    print("\n\n")

Batch(id='batch_68319ffd2864819085fe96a618581c0a', completion_window='24h', created_at=1748082685, endpoint='/v1/chat/completions', input_file_id='file-DbgthxrE99JzWKtqSjahkw', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1748169085, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 't_kewrag_only_errors_vs_rag.jsonl_3'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))





## Creating the Batch

Once you've successfully uploaded your input file, you can use the input File object's ID to create a batch. For now, the completion window can only be set to 24h. You can also provide custom metadata via an optional metadata parameter.

In [ ]:
# # Get the ID of the bad batch input file
# batch_input_file_id = batch_input_file.id

# # Create a batch job using the bad batch input file
# batch = client.batches.create(
#     input_file_id=batch_input_file_id,  # ID of the input file for the batch
#     endpoint="/v1/chat/completions",  # API endpoint to use for the batch
#     completion_window="24h",  # Time window for completion
#     metadata={
#         "description": "test"  # Metadata describing the batch job
#     }
# )

# # Print the details of the bad batch job
# print(batch)
# print("\n\n")

## Checking Batch Status

The status of a given Batch object can be any of the following:

| STATUS      | DESCRIPTION                                                                  |
|-------------|------------------------------------------------------------------------------|
| validating  | the input file is being validated before the batch can begin                 |
| failed      | the input file has failed the validation process                             |
| in_progress | the input file was successfully validated and the batch is currently being run |
| finalizing  | the batch has completed and the results are being prepared                   |
| completed   | the batch has been completed and the results are ready                       |
| expired     | the batch was not able to be completed within the 24-hour time window        |
| cancelling  | the batch is being cancelled (may take up to 10 minutes)                     |
| cancelled   | the batch was cancelled                                                      |


In [31]:
[i.id for i in list(client.batches.list())]

['batch_68319ffd2864819085fe96a618581c0a',
 'batch_68319eb47194819095cba61249ce9fec',
 'batch_68319daee3248190beaed7fe0822ac2f',
 'batch_68319b85ae048190a3a14292e67c485e',
 'batch_6831927867b0819095bb50feb626e8f1',
 'batch_68319272f2a881908ece3e68d8aa2a5f',
 'batch_6831926954dc81908efedbbab50cd374',
 'batch_6831916dd2cc8190bd9aafd9b2c6c2f7',
 'batch_68318b78923c8190b9ffa4b857666974',
 'batch_68318b73016c8190b1ee9c8db1670299',
 'batch_68318b68295c8190be5a7b0c3ae85d63',
 'batch_68318b659a748190a3f3a3c6c4a9bd51',
 'batch_68318492c3c48190b72029f53bcff60c',
 'batch_68318130158881909a0c3da64591a7a9',
 'batch_68317c0d36c08190bc2296ad055afb3f',
 'batch_6831796ac5848190809216a0b58c4735',
 'batch_683178befe1c81908ed6772a5027b1bf',
 'batch_683175aeb9488190bb670cd6a09e720d',
 'batch_683171f404b08190aa8f61f373b1d79f']

In [ ]:
batches = list(client.batches.list())

# Print the total number of batches
print("\nWe have " + str(len(batches)) + " batches\n")

# Iterate over the batches and print their ids, statuses, and descriptions
for batch in batches:
    print(batch.id)  # Print the batch ID
    print(batch.status)  # Print the batch status
    print(batch.metadata.get("description"))  # Print the batch description from metadata
    print("\n\n")  # Print newline characters for better readability

In [ ]:
# Retrieve and print the details of the bad batch job
print(client.batches.retrieve(batch.id))
print("\n\n")  # Print newline characters for better readability


## Cancelling a Batch

If necessary, you can cancel an ongoing batch. The batch's status will change to cancelling until in-flight requests are complete (up to 10 minutes), after which the status will change to cancelled.

In [ ]:
# dead_batch_walking = client.batches.create(
#     input_file_id=batch_input_file_id,
#     endpoint="/v1/chat/completions",
#     completion_window="24h",
#     metadata={
#         "description": "good nightly penguin job"
#     }
# )

# print(dead_batch_walking)
# time.sleep(5)
# print("\n\n")
# print(client.batches.retrieve(dead_batch_walking.id))
# print("\n\n")
for batch_id in [i.id for i in list(client.batches.list())]:
    print(client.batches.cancel(batch_id))
# time.sleep(15)
# print("\n\n")
# print(client.batches.retrieve(dead_batch_walking.id))

## Getting the Results

Once the batch is complete, you can download the output by making a request against the Files API via the output_file_id field from the Batch object and writing it to a file on your machine, in this case batch_output.jsonl

In [ ]:
client.batches.retrieve(batch.id)

In [ ]:
# Fetch the content of the file
content = client.files.content(client.batches.retrieve(batch.id).output_file_id)

content.write_to_file("output.jsonl")


In [10]:
import json

In [34]:
file_path = 'result_batch/t_zscot_only_errors_vs_kewltm_batch_68319b85ae048190a3a14292e67c485e_output.jsonl' 

t_zscot_only_errors_vs_kewltm_batch = {}

with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        
        data_id = data['custom_id']
        data_content = data['response']['body']['choices'][0]['message']['content']
        data_content = json.loads(data_content)
        if data_content['error_cause'] == "Numerical Incompetence":
            print(data_id)
            print(data_content)
            print("--------------------")

        t_zscot_only_errors_vs_kewltm_batch[data_id] = data_content

print("The number of 'ZSCOT' only errors (vs 'KEwLTM') :", end=" ")
print(len(t_zscot_only_errors_vs_kewltm_batch))

error_cause_counter = {
    "Incorrect Information Extraction": 0,
    "Incorrect Inference": 0,
    "Numerical Incompetence": 0,
    "Incorrect Knowledge": 0,
    "Conflicting Ground Truth": 0,
    "Incomplete Information": 0
}
for data_id, data_content in t_zscot_only_errors_vs_kewltm_batch.items():
    error_cause = data_content['error_cause']
    error_cause_counter[error_cause] += 1
print(error_cause_counter)

request-TCGA-A2-A0ST.1E8B978E-3D1B-46E8-80EE-3A11EE571CEC-4
{'error_cause': 'Numerical Incompetence', 'reason': 'The model correctly extracted the tumor size (2.0 cm) but then stated that this value meets the T2 criterion of “> 2.0 cm,” mistakenly treating 2.0 cm as greater than 2.0 cm. The AJCC rule itself was cited correctly (T2 is >2 cm), so the fault lies in the numeric comparison, not in AJCC knowledge.'}
--------------------
request-TCGA-A2-A0YL.AB083371-A2AB-4F57-8FBF-0DB2C91CEDDE-5
{'error_cause': 'Numerical Incompetence', 'reason': 'The model correctly extracted the tumor size as 9 cm and even restated the AJCC rule that tumors "more than 5 cm" are T3, but it still concluded T2. The facts and staging rule were present; the error came from an illogical numerical comparison (treating 9 cm as ≤5 cm).'}
--------------------
request-TCGA-A8-A06T.5D888D65-685F-4598-884F-AD057DE5FD1D-7
{'error_cause': 'Numerical Incompetence', 'reason': 'The model correctly extracted the tumor size (

In [ ]:
file_path = 'result_batch/t_kewltm_only_errors_vs_zscot_batch_68319daee3248190beaed7fe0822ac2f_output.jsonl' 
    
t_kewltm_only_errors_vs_zscot_batch = {}

with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        
        data_id = data['custom_id']
        data_content = data['response']['body']['choices'][0]['message']['content']
        data_content = json.loads(data_content)

        t_kewltm_only_errors_vs_zscot_batch[data_id] = data_content

print("The number of 'KEwLTM' only errors (vs 'ZSCOT') :", end=" ")
print(len(t_kewltm_only_errors_vs_zscot_batch))

error_cause_counter = {
    "Incorrect Information Extraction": 0,
    "Incorrect Inference": 0,
    "Numerical Incompetence": 0,
    "Incorrect Knowledge": 0,
    "Conflicting Ground Truth": 0,
    "Incomplete Information": 0
}
for data_id, data_content in t_kewltm_only_errors_vs_zscot_batch.items():
    error_cause = data_content['error_cause']
    error_cause_counter[error_cause] += 1
print(error_cause_counter)


The number of 'KEwLTM' only errors (vs 'ZSCOT') : 26
{'Incorrect Information Extraction': 4, 'Incorrect Inference': 0, 'Numerical Incompetence': 19, 'Incorrect Knowledge': 2, 'Conflicting Ground Truth': 1, 'Incomplete Information': 0}


In [29]:
file_path = 'result_batch/t_rag_only_errors_vs_kewrag_batch_68319eb47194819095cba61249ce9fec_output.jsonl' 

t_rag_only_errors_vs_kewrag_batch = {}

with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        
        data_id = data['custom_id']
        data_content = data['response']['body']['choices'][0]['message']['content']
        data_content = json.loads(data_content)

        t_rag_only_errors_vs_kewrag_batch[data_id] = data_content

print("The number of 'RAG' only errors (vs 'KEwRAG') :", end=" ")
print(len(t_rag_only_errors_vs_kewrag_batch))

error_cause_counter = {
    "Incorrect Information Extraction": 0,
    "Incorrect Inference": 0,
    "Numerical Incompetence": 0,
    "Incorrect Knowledge": 0,
    "Conflicting Ground Truth": 0,
    "Incomplete Information": 0
}
for data_id, data_content in t_rag_only_errors_vs_kewrag_batch.items():
    error_cause = data_content['error_cause']
    error_cause_counter[error_cause] += 1
print(error_cause_counter)


The number of 'RAG' only errors (vs 'KEwRAG') : 81
{'Incorrect Information Extraction': 11, 'Incorrect Inference': 5, 'Numerical Incompetence': 53, 'Incorrect Knowledge': 11, 'Conflicting Ground Truth': 1, 'Incomplete Information': 0}


In [33]:
file_path = 'result_batch/t_kewrag_only_errors_vs_rag_batch_68319ffd2864819085fe96a618581c0a_output.jsonl' 
    
t_kewrag_only_errors_vs_rag_batch = {}

with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        
        data_id = data['custom_id']
        data_content = data['response']['body']['choices'][0]['message']['content']
        data_content = json.loads(data_content)
        if data_content['error_cause'] == "Incorrect Information Extraction":
            print(data_id)
            print(data_content['reason'])
            print("--------")

        t_kewrag_only_errors_vs_rag_batch[data_id] = data_content

print("The number of 'KEwRAG' only errors (vs 'RAG') :", end=" ")
print(len(t_kewrag_only_errors_vs_rag_batch))

error_cause_counter = {
    "Incorrect Information Extraction": 0,
    "Incorrect Inference": 0,
    "Numerical Incompetence": 0,
    "Incorrect Knowledge": 0,
    "Conflicting Ground Truth": 0,
    "Incomplete Information": 0
}
for data_id, data_content in t_kewrag_only_errors_vs_rag_batch.items():
    error_cause = data_content['error_cause']
    error_cause_counter[error_cause] += 1
print(error_cause_counter)


request-TCGA-3C-AALI.84E6A935-1A49-4BC1-9669-3DEA161CF6FC-0
The report clearly states “GROSS/MICRO FINAL INVASIVE TUMOR SIZE … 3.2 x 3.0 x 3.0 cm,” which exceeds 2 cm and makes the tumor T2. In its reasoning the model applies the correct AJCC size rules but still outputs T1, showing it never incorporated the 3.2 cm measurement; it must have missed or mis-read the key size information rather than mis-applying the staging rules.
--------
request-TCGA-A2-A1G4.5AD3B697-F097-496C-978B-F73BDA5394A7-4
The report clearly states a right-breast invasive carcinoma “spanning a distance of 65 mm,” which meets T3 (>5 cm). The model ignored this larger tumor and focused only on the 21 mm left-breast lesion, so it never considered the decisive fact for staging.
--------
request-TCGA-AC-A2QJ.E422C6E8-084F-42B8-84E7-4EF387F098C7-11
The report clearly states that the invasive carcinoma invades the dermis with overlying skin ulceration and explicitly labels the tumor pT4b. The model’s explanation only cit